# 0. Environment preparation
## 0.1. Importing libraries

In [191]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
from IPython.display import clear_output

\-

\-

\-

\-

# 1. Model Loading

In [192]:
def load_model():
    filename = "predictor.pkl"

    if os.path.exists(filename):
        model = joblib.load(filename)["model"]
        print(f"\nLoaded Joblib model: {filename}")
        return model, "joblib"

    raise FileNotFoundError(
        "Could not find predictor.pkl"
    )

\-

\-

\-

\-

# 2. Feature Engineering

In [193]:
def extract_city(user_data):
    user_data['displayAddress'] = "," + user_data['displayAddress']
    return user_data['displayAddress'].str.split(',', n=1).str[1].str.strip().to_frame() # Extracts the last part of the string, which is the area and the city.


def build_input_dataframe(user_data):
    EPS = 1e-6

    furnished = user_data["furnished"].upper()

    furnished_no = 1 if furnished == "N" else 0
    furnished_partly = 1 if furnished == "P" else 0
    furnished_yes = 1 if furnished == "Y" else 0

    bedrooms = float(user_data["bedrooms"])
    bathrooms = float(user_data["bathrooms"])
    size_min = float(user_data["sizeMin"])

    df = pd.DataFrame({
        "displayAddress": [user_data["areaAndCity"]],
        "bathrooms": [bathrooms],
        "bedrooms": [bedrooms],
        "verified": [1 if user_data["verified"].upper() == "Y" else 0],
        "sizeMin": [size_min],
        "month_listed": [int(user_data["month_listed"])],
        "furnished_NO": [furnished_no],
        "furnished_PARTLY": [furnished_partly],
        "furnished_YES": [furnished_yes],

        # engineered features
        "bathroom_bedroom_ratio":
            [bathrooms / (bedrooms + EPS)],

        "size_per_bedroom":
            [size_min / (bedrooms + EPS)],

        "size_per_bathroom":
            [size_min / (bathrooms + EPS)],

        "bedrooms_per_size":
            [bedrooms / (size_min + EPS)],

        "bedrooms_bathrooms_interaction":
            [bedrooms * bathrooms]
    })

    # Preserve exact column order
    df = df[[
        "displayAddress",
        "bathrooms",
        "bedrooms",
        "verified",
        "sizeMin",
        "month_listed",
        "furnished_NO",
        "furnished_PARTLY",
        "furnished_YES",
        "bathroom_bedroom_ratio",
        "size_per_bedroom",
        "size_per_bathroom",
        "bedrooms_per_size",
        "bedrooms_bathrooms_interaction"
    ]]

    return df

\-

\-

\-

\-

# 3. Helper Functions

In [194]:
def quit_if_requested(value):
    if value.strip().lower() == "q":
        print("\nExiting program...\nGoodbye.")
        sys.exit(0)

In [195]:
def get_input(prompt, isnumeric, default=""):
    while True:
        value = input(prompt).strip()

        if default != "" and value == "":
            value = default
            break

        elif value == "":
            print("Input cannot be empty. Please try again.")
            continue

        elif isnumeric and value.lower() == "q":
            break

        elif isnumeric and not value.replace("-", "").isdigit():
            print("Input must be a number. Please try again.")
            continue

        elif isnumeric and value.replace("-", "", 1).isdigit() and int(value) < 0:
            print("Input must be a positive number. Please try again.")
            continue

        elif isnumeric and value.isdigit() and int(value) >= 0:
            break

        elif not isnumeric and value.lower() not in ["y", "n", "p", "q"]:
            print("Invalid input. Please try again.")
            continue

        elif not isnumeric and value.lower() in ["y", "n", "p", "q"]:
            break

    return value

\-

\-

\-

\-

# 4. Getting User Input

In [196]:
def get_property_input(existing=None):

    if existing is None:
        existing = {}

    print("\nEnter Property Information")
    print("-" * 30)

    areaAndCity = (
        input(f"Area and City [{existing.get('areaAndCity','')}]: ")
        or existing.get("areaAndCity", "")
    )
    print(f"{'-> Area and City: ':<20}{areaAndCity}")


    bathrooms = get_input(
        f"Bathrooms [{existing.get('bathrooms','')}]: ",
        isnumeric=True,
        default=existing.get('bathrooms','')
    )

    print(f"{'-> Bathrooms: ':<20}{bathrooms}")
    quit_if_requested(bathrooms)


    bedrooms = get_input(
        f"Bedrooms [{existing.get('bedrooms','')}]: ",
        isnumeric=True,
        default=existing.get('bedrooms','')
    )
    print(f"{'-> Bedrooms: ':<20}{bedrooms}")
    quit_if_requested(bedrooms)

    verified = get_input(
        f"Verified (Y/N) [{existing.get('verified','')}]: ",
        isnumeric=False,
        default=existing.get('verified','')
    )
    print(f"{'-> Verified: ':<20}{verified}")
    quit_if_requested(verified)

    sizeMin = get_input(
        f"Size (sqft) [{existing.get('sizeMin','')}]: ",
        isnumeric=True,
        default=existing.get('sizeMin','')
    )
    print(f"{'-> Size (sqft): ':<20}{sizeMin}")
    quit_if_requested(sizeMin)

    month_listed = get_input(
        f"Month Listed (1-12) [{existing.get('month_listed','')}]: ",
        isnumeric=True,
        default=existing.get('month_listed','')
    )
    print(f"{'-> Month Listed (1-12): ':<20}{month_listed}")
    quit_if_requested(month_listed)

    furnished = get_input(
        f"Furnished (Y/N/P) [{existing.get('furnished','')}]: ",
        isnumeric=False,
        default=existing.get('furnished','')
    )
    print(f"{'-> Furnished (Y/N/P): ':<20}{furnished}")
    quit_if_requested(furnished)

    return {
        "areaAndCity": areaAndCity,
        "bathrooms": bathrooms,
        "bedrooms": bedrooms,
        "verified": verified,
        "sizeMin": sizeMin,
        "month_listed": month_listed,
        "furnished": furnished
    }

\-

\-

\-

\-

# 5. Prediction Function

In [197]:
def predict_price(model, property_data):

    features = build_input_dataframe(property_data)

    log_prediction = model.predict(features)

    predicted_price = np.expm1(log_prediction[0])

    print("\n================================")
    print(f"Predicted Price: AED {predicted_price:,.0f}")
    print("================================\n")

    return predicted_price

\-

\-

\-

\-

# 6. Main Menu

In [198]:
def main():

    model, model_type = load_model()

    last_property = None

    while True:
        print("\nMenu")
        print("1. New Prediction")
        print("2. Edit Last Prediction")
        print("3. Exit")

        choice = input("\nSelect option: ").strip()

        clear_output(wait=True)

        if choice == "1":

            last_property = get_property_input()
            predict_price(model, last_property)

        elif choice == "2":

            if last_property is None:
                print("\nNo previous prediction available.")
                continue

            print("\nPress ENTER to keep existing values.")

            last_property = get_property_input(last_property)
            predict_price(model, last_property)

        elif choice == "3":

            print("\nGoodbye.")
            break

        else:
            print("\nInvalid option.")

        


try:
    if __name__ == "__main__":
        main()
except:
    None


Goodbye.
